In [1]:
import sys
from pathlib import Path
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "model.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.data import ORDERS_PATH, TARGET, load_raw
from src.model import FEATURES, NUMERIC, time_split, make_logistic, make_forest, make_boosting, score

orders = pd.read_csv(ORDERS_PATH, parse_dates=["order_date"])
train, test = time_split(orders)
mode_rate = train.groupby("shipping_mode")[TARGET].mean()
results = [score("shipping-mode lookup", test[TARGET], test["shipping_mode"].map(mode_rate))]

In [2]:
raw = load_raw()
real_days = raw.groupby("Order Id")["Days for shipping (real)"].first().rename("real_days")
leaky_train = train.join(real_days, on="Order Id")
leaky_test = test.join(real_days, on="Order Id")

cheat = make_logistic(numeric=NUMERIC + ["real_days"])
cheat.fit(leaky_train[FEATURES + ["real_days"]], leaky_train[TARGET])
cheat_prob = cheat.predict_proba(leaky_test[FEATURES + ["real_days"]])[:, 1]
print(score("CHEATING: + real shipping days", test[TARGET], cheat_prob))

{'model': 'CHEATING: + real shipping days', 'accuracy': 1.0, 'roc_auc': 1.0}


In [3]:
probs = {}
for name, make in [("logistic regression", make_logistic),
                   ("random forest", make_forest),
                   ("gradient boosting", make_boosting)]:
    model = make().fit(train[FEATURES], train[TARGET])
    probs[name] = model.predict_proba(test[FEATURES])[:, 1]
    results.append(score(name, test[TARGET], probs[name]))
pd.DataFrame(results)

,model,accuracy,roc_auc
0,shipping-mode lookup,0.6953,0.7443
1,logistic regression,0.7062,0.7528
2,random forest,0.7251,0.7754
3,gradient boosting,0.7223,0.7773


In [4]:
scored = test.assign(prob=probs["gradient boosting"])
for mode, part in scored.groupby("shipping_mode"):
    if part[TARGET].nunique() < 2:
        print(f"{mode:15} n={len(part):5} | always {part[TARGET].iloc[0]}, nothing to rank")
        continue
    auc = roc_auc_score(part[TARGET], part["prob"])
    print(f"{mode:15} n={len(part):5} | late {part[TARGET].mean():.3f} | AUC within this mode: {auc:.3f}")

First Class     n= 1900 | always 1, nothing to rank
Same Day        n=  702 | late 0.533 | AUC within this mode: 1.000
Second Class    n= 2470 | late 0.802 | AUC within this mode: 0.494
Standard Class  n= 7508 | late 0.396 | AUC within this mode: 0.505


In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text

same = orders[orders["shipping_mode"] == "Same Day"]
X = pd.get_dummies(same[FEATURES], dtype=int)      # raw numbers kept, so the rule is readable
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X, same[TARGET])

print(export_text(tree, feature_names=list(X.columns)))
print("accuracy of this 2-level tree on Same Day orders:", round(tree.score(X, same[TARGET]), 4))

|--- order_hour <= 11.50
|   |--- class: 0
|--- order_hour >  11.50
|   |--- class: 1

accuracy of this 2-level tree on Same Day orders: 1.0


In [6]:
shipped = raw[raw["Delivery Status"] != "Shipping canceled"]
o = orders.join(shipped.groupby("Order Id")["Days for shipping (real)"].first().rename("real_days"), on="Order Id")
print(pd.crosstab(o["shipping_mode"], o["real_days"]))
print()
print(o.groupby("shipping_mode")["scheduled_days"].first())

real_days          0     1     2     3     4     5     6
shipping_mode                                           
First Class        0     0  9602     0     0     0     0
Same Day        1759  1648     0     0     0     0     0
Second Class       0     0  2453  2422  2473  2452  2456
Standard Class     0     0  7610  7517  7510  7458  7537

shipping_mode
First Class       1
Same Day          0
Second Class      2
Standard Class    4
Name: scheduled_days, dtype: int64


In [7]:
def late_rate_under(o, promise: dict, same_day_cutoff=None) -> float:
    p = o["shipping_mode"].map(promise)
    if same_day_cutoff is not None:          # Same Day orders after the cutoff get a next-day promise
        after = (o["shipping_mode"] == "Same Day") & (o["order_hour"] >= same_day_cutoff)
        p = p.where(~after, 1)
    return (o["real_days"] > p).mean()

today = {"First Class": 1, "Same Day": 0, "Second Class": 2, "Standard Class": 4}
fc2 = {**today, "First Class": 2}
scenarios = {
    "today":                                     (today, None),
    "First Class promises 2 days":               (fc2, None),
    "+ Same Day after noon becomes next-day":    (fc2, 12),
    "+ Standard Class promises 5 days":          ({**fc2, "Standard Class": 5}, 12),
    "+ Second Class promises 5 days":            ({**fc2, "Standard Class": 5, "Second Class": 5}, 12),
}
pd.DataFrame([{"scenario": name, "late_rate": round(late_rate_under(o, *args), 3)}
              for name, args in scenarios.items()])

,scenario,late_rate
0,today,0.573
1,First Class promises 2 days,0.420
2,+ Same Day after noon becomes next-day,0.394
3,+ Standard Class promises 5 days,0.276
4,+ Second Class promises 5 days,0.159
